# ★ Capstone: Guardian Angel System (GAS)
## Complete Multi-Agent Diabetic Patient Supervision System

> **Book:** *Building Agentic AI: From First Principles to a Multi-Agent Guardian Angel System*  
> **Chapter:** Capstone (★) — Final Integration  
> **Prerequisites:** All 28 chapters (or just run the cells — everything is self-contained)

---

⚠️ **Clinical Disclaimer:** This notebook is for educational purposes only. The GAS system is **NOT a medical device** and must **NOT** be used for clinical decision-making or patient care. Always consult a qualified healthcare professional.

---

## What You'll Learn

This capstone notebook brings together every concept from the book into a single, running system:

1. **The Journey** — What we built across all 28 chapters
2. **GAS Architecture** — Deep dive into the full system design
3. **Running GAS** — Execute all 5 clinical scenarios
4. **Results Analysis** — Visualise glucose traces, alert metrics, latency
5. **Extending GAS** — How to add agents, tools, and deploy to production
6. **Clinical Considerations** — What GAS does well, what it cannot do, ethics


---
## Setup

In [ ]:
# Install dependencies (skip if already installed)
!pip install -r requirements.txt

import os
import sys
import time
import warnings
import requests
warnings.filterwarnings('ignore')

# Add parent directory to path for shared utilities
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'agentic-ai-book'))

# Detect available backend — prefer live LLM over mock
def _ollama_reachable(url='http://localhost:11434'):
    try:
        return requests.get(f'{url}/api/tags', timeout=2).status_code == 200
    except Exception:
        return False

if os.environ.get('NOTEBOOK_TEST_MODE') == '1':
    print('ℹ️  NOTEBOOK_TEST_MODE=1 — running in mock mode')
elif os.environ.get('OPENAI_API_KEY'):
    print('✅ OpenAI API key detected — using OpenAI backend')
elif os.environ.get('ANTHROPIC_API_KEY'):
    print('✅ Anthropic API key detected — using Anthropic backend')
elif _ollama_reachable():
    print('✅ Ollama reachable — using local Ollama backend')
else:
    os.environ['NOTEBOOK_TEST_MODE'] = '1'
    print('ℹ️  No API keys or Ollama found — running in NOTEBOOK_TEST_MODE (mock responses)')

print('Setup complete.')

In [ ]:
# Core imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from datetime import datetime, timezone

# GAS system
from gas_system import (
    PatientProfile, GlucoseReading, GASState,
    CGMSimulator, GASKnowledgeBase, GASLLMClient,
    PATIENT_PROFILES, MEDICAL_KNOWLEDGE,
    monitor_node, advisor_node, safety_node, caregiver_node, supervisor_node,
    build_gas_graph, run_simulation, get_patient,
    TEST_MODE,
)

# Ensure figures directory exists (needed for savefig calls)
os.makedirs('figures', exist_ok=True)

print(f'GAS system loaded. TEST_MODE={TEST_MODE}')
print(f'Available patients: {list(PATIENT_PROFILES.keys())}')
print(f'Available scenarios: {CGMSimulator.SCENARIOS}')

---
# 🎯 Section 1 — The Journey: 28 Chapters to a Guardian Angel

This book took you from first principles to a production-quality multi-agent AI system. Here's what we built in each chapter:

In [ ]:
CHAPTER_JOURNEY = [
    # (chapter, title, concept, gas_piece)
    (1,  'The Agentic AI Landscape',        'Agents, environments, reward signals',         'Project scaffold & shared utilities'),
    (2,  'LLM Foundations for RL',          'Tokenisation, next-token prediction, RLHF',    'PatientProfile & GlucoseReading data models'),
    (3,  'MDPs for LLMs',                   'MDPs, policies, value functions',              'GASState TypedDict — the MDP state'),
    (4,  'Reward Modelling',                'Bradley-Terry, preference datasets',           'Reward model for clinical advice quality'),
    (5,  'Proximal Policy Optimisation',    'PPO algorithm, clipping, KL penalty',          'PPO fine-tune loop for the Advisor agent'),
    (6,  'Direct Preference Optimisation',  'DPO loss, reference model, β parameter',       'DPO alternative for Advisor; A/B comparison'),
    (7,  'GRPO & Group Relative Rewards',   'GRPO, group baselines, variance reduction',    'GRPO fine-tune for Safety Critic scoring'),
    (8,  'Constitutional AI & RLAIF',       'Critique-revision, AI feedback loops',         'Safety Critic constitutional rules'),
    (9,  'Reasoning & Chain-of-Thought',    'CoT, scratchpads, process reward models',      'Monitor reasoning trace for anomaly detection'),
    (10, 'Tree-of-Thought & Search',        'ToT, beam search, MCTS for LLMs',             'Supervisor planning tree'),
    (11, 'Tool Use & Function Calling',     'OpenAI tools API, JSON schema',               'Tool registry for Monitor (CGM API, meal DB)'),
    (12, 'Retrieval-Augmented Generation',  'Dense retrieval, re-ranking, Qdrant',          'GASKnowledgeBase — FAISS + sentence-transformers'),
    (13, 'Memory Systems for Agents',       'Episodic, semantic, working memory',           'readings_history rolling window in GASState'),
    (14, 'LangChain Fundamentals',          'Chains, runnables, LCEL',                     'LangChain wiring for Monitor + Advisor'),
    (15, 'LangGraph: Stateful Graphs',      'Nodes, edges, conditional routing',           'build_gas_graph() — the full StateGraph'),
    (16, 'CrewAI: Role-Based Teams',        'Crews, roles, tasks, delegation',             'CrewAI crew wrapping all 5 GAS agents'),
    (17, 'AutoGen: Conversational Agents',  'GroupChat, AssistantAgent, UserProxy',         'AutoGen fallback orchestration layer'),
    (18, 'Model Context Protocol (MCP)',    'MCP spec, tool servers, resources',            'MCP server exposing GAS tools'),
    (19, 'Evaluation Frameworks',           'LLM-as-judge, G-Eval, RAGAS, FactScore',      'Evaluation harness for Advisor responses'),
    (20, 'Safety, Alignment, Red-Teaming',  'Jailbreaks, prompt injection, adversarial',    'Red-team suite for Safety Critic'),
    (21, 'Observability & Tracing',         'LangSmith, OpenTelemetry, structured logs',    'Tracing middleware for all GAS agents'),
    (22, 'Streaming & Real-Time Agents',    'SSE, WebSockets, async generators',           'Real-time glucose alert streaming'),
    (23, 'Vector Databases Deep Dive',      'HNSW, quantisation, filtering, Qdrant',        'Production Qdrant setup for Advisor RAG'),
    (24, 'Fine-Tuning with PEFT & LoRA',    'LoRA, QLoRA, adapter merging',                'LoRA adapter for domain-adapted Advisor'),
    (25, 'Deploying Agents to Production',  'FastAPI, Docker, health checks',              'Dockerised GAS microservice deployment'),
    (26, 'Human-in-the-Loop Patterns',      'Interrupt nodes, approval workflows',          'HITL approval gate in Supervisor graph'),
    (27, 'Scaling Multi-Agent Systems',     'Horizontal scaling, message queues',           'Redis Streams task queue for GAS'),
    (28, 'Future Directions',               'World models, embodied agents, AGI safety',    'Research roadmap notebook'),
]

df_journey = pd.DataFrame(CHAPTER_JOURNEY, columns=['Chapter', 'Title', 'Key Concept', 'GAS Piece Built'])
print(f'Total chapters: {len(df_journey)}')
df_journey.set_index('Chapter')

In [ ]:
# Visualise the journey as a timeline
fig, ax = plt.subplots(figsize=(14, 10))

parts = [
    ('Part 1: Foundations',    1,  4,  '#4CAF50'),
    ('Part 2: RL Methods',     5,  8,  '#2196F3'),
    ('Part 3: Reasoning',      9,  13, '#FF9800'),
    ('Part 4: Evaluation',     19, 21, '#9C27B0'),
    ('Part 5: Agentic',        14, 18, '#F44336'),
    ('Part 5: Agentic (cont)', 22, 28, '#F44336'),
]

y_positions = {ch: i for i, (ch, _, _, _) in enumerate(CHAPTER_JOURNEY)}

for i, (ch, title, concept, piece) in enumerate(CHAPTER_JOURNEY):
    # Find part color
    color = '#607D8B'
    for part_name, start, end, c in parts:
        if start <= ch <= end:
            color = c
            break
    
    ax.barh(i, 1, left=0, height=0.7, color=color, alpha=0.8)
    ax.text(1.05, i, f'Ch.{ch:02d}: {title}', va='center', fontsize=8)
    ax.text(-0.05, i, f'→ {piece[:50]}', va='center', ha='right', fontsize=7, color='#333')

# Capstone star
ax.barh(len(CHAPTER_JOURNEY), 1, left=0, height=0.7, color='gold', alpha=1.0)
ax.text(1.05, len(CHAPTER_JOURNEY), '★ Capstone: Complete Guardian Angel System', 
        va='center', fontsize=9, fontweight='bold')

# Legend
legend_patches = [mpatches.Patch(color=c, label=name) for name, _, _, c in parts]
legend_patches.append(mpatches.Patch(color='gold', label='★ Capstone'))
ax.legend(handles=legend_patches, loc='lower right', fontsize=8)

ax.set_xlim(-3, 3)
ax.set_ylim(-1, len(CHAPTER_JOURNEY) + 1)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title('The Journey: 28 Chapters → Guardian Angel System', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

plt.tight_layout()
plt.savefig('figures/chapter_journey.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to figures/chapter_journey.png')

---
# 📖 Section 2 — GAS Architecture Deep Dive

Let's explore each component of the Guardian Angel System in detail.

In [ ]:
# 2.1 Patient Profiles
print('=== Patient Profiles ===')
for name, patient in PATIENT_PROFILES.items():
    print(f'\n{name.upper()}: {patient.name}')
    print(f'  Age: {patient.age}, Type {patient.diabetes_type} Diabetes')
    print(f'  Target range: {patient.target_low}–{patient.target_high} mg/dL')
    print(f'  ISF: {patient.insulin_sensitivity_factor} mg/dL/unit')
    print(f'  Medications: {patient.medications}')
    print(f'  Caregiver: {patient.caregiver_name} ({patient.caregiver_email})')

In [ ]:
# 2.2 CGM Simulator — visualise all 5 scenarios
simulator = CGMSimulator(seed=42)

fig, axes = plt.subplots(5, 1, figsize=(14, 18), sharex=True)
fig.suptitle('CGM Simulator — All 5 Clinical Scenarios (24-hour traces)', 
             fontsize=14, fontweight='bold', y=1.01)

scenario_colors = {
    'normal_day':           '#4CAF50',
    'hypoglycemia_episode': '#F44336',
    'post_meal_spike':      '#FF9800',
    'dawn_phenomenon':      '#9C27B0',
    'exercise_induced_low': '#2196F3',
}

scenario_titles = {
    'normal_day':           'Normal Day — Typical 24-hour profile',
    'hypoglycemia_episode': 'Hypoglycemia Episode — Nocturnal low at 02:00',
    'post_meal_spike':      'Post-Meal Spike — Large lunch spike to ~280 mg/dL',
    'dawn_phenomenon':      'Dawn Phenomenon — Early-morning glucose rise',
    'exercise_induced_low': 'Exercise-Induced Low — Aerobic exercise at 17:00',
}

all_readings = {}
for ax, scenario in zip(axes, CGMSimulator.SCENARIOS):
    readings = simulator.generate(scenario)
    all_readings[scenario] = readings
    
    times = [(r.timestamp.hour + r.timestamp.minute/60) for r in readings]
    values = [r.value_mgdl for r in readings]
    
    # Target range shading
    ax.axhspan(70, 180, alpha=0.1, color='green', label='Target range (70–180)')
    ax.axhline(70, color='orange', linestyle='--', linewidth=1, alpha=0.7, label='Low threshold (70)')
    ax.axhline(54, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Critical low (54)')
    ax.axhline(180, color='orange', linestyle='--', linewidth=1, alpha=0.5, label='High threshold (180)')
    ax.axhline(250, color='red', linestyle='--', linewidth=1, alpha=0.5, label='High alert (250)')
    
    # Glucose trace
    ax.plot(times, values, color=scenario_colors[scenario], linewidth=2, label='Glucose')
    
    # Highlight critical readings
    critical_times = [t for t, v in zip(times, values) if v < 54 or v > 400]
    critical_vals = [v for v in values if v < 54 or v > 400]
    if critical_times:
        ax.scatter(critical_times, critical_vals, color='red', s=50, zorder=5, label='Critical')
    
    ax.set_ylabel('mg/dL', fontsize=9)
    ax.set_ylim(30, 320)
    ax.set_title(scenario_titles[scenario], fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Stats annotation
    avg = np.mean(values)
    tir = sum(1 for v in values if 70 <= v <= 180) / len(values) * 100
    tbr = sum(1 for v in values if v < 70) / len(values) * 100
    ax.text(23.5, 290, f'Avg: {avg:.0f}\nTIR: {tir:.0f}%\nTBR: {tbr:.0f}%', 
            ha='right', va='top', fontsize=8, 
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

axes[-1].set_xlabel('Hour of Day', fontsize=10)
axes[-1].set_xticks(range(0, 25, 2))
axes[-1].set_xticklabels([f'{h:02d}:00' for h in range(0, 25, 2)], rotation=45)

plt.tight_layout()
plt.savefig('figures/all_scenarios_glucose.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to figures/all_scenarios_glucose.png')

In [ ]:
# 2.3 RAG Knowledge Base
print('=== GAS Knowledge Base ===')
print(f'Total documents: {len(MEDICAL_KNOWLEDGE)}')
print()

kb = GASKnowledgeBase()
kb.build_index()

# Test retrieval
test_queries = [
    'patient has glucose 48 mg/dL falling fast at night',
    'glucose 280 mg/dL after lunch rising trend',
    'early morning glucose rise dawn phenomenon',
]

for query in test_queries:
    print(f'Query: "{query}"')
    docs = kb.retrieve(query, k=2)
    for i, doc in enumerate(docs):
        print(f'  [{i+1}] {doc[:120]}...')
    print()

In [ ]:
# 2.4 LLM Client
llm = GASLLMClient()
print(f'LLM Client: {llm}')

# Test the LLM
test_messages = [
    {'role': 'system', 'content': 'You are the GAS Advisor agent. Be concise.'},
    {'role': 'user', 'content': 'Patient glucose is 52 mg/dL, falling fast. What should they do?'},
]
response = llm.chat(test_messages, temperature=0.3)
print(f'\nLLM Response:\n{response}')

In [ ]:
# 2.5 LangGraph — visualise the graph structure
print('=== LangGraph Structure ===')

try:
    gas_graph = build_gas_graph()
    print('Graph compiled successfully!')
    print()
    
    # Try to get graph representation
    try:
        graph_repr = gas_graph.get_graph()
        print('Nodes:', list(graph_repr.nodes.keys()))
        print('Edges:', [(e.source, e.target) for e in graph_repr.edges])
    except Exception:
        print('Nodes: supervisor, monitor, advisor, safety, caregiver')
        print('Edges: START→monitor, monitor→supervisor, advisor→supervisor,')
        print('       safety→supervisor, caregiver→END')
        print('Conditional: supervisor→{advisor, safety, caregiver, END}')
        
except ImportError as e:
    print(f'LangGraph not installed: {e}')
    print('Install with: pip install langgraph')

---
# 🔨 Section 3 — Running GAS: All 5 Scenarios

Let's run the full Guardian Angel System on all 5 clinical scenarios.

In [ ]:
# Run all 5 scenarios
# Note: In TEST_MODE, this uses mock LLM responses (fast, no API calls)
# With a real API key, this will call the LLM for each reading

# For the notebook, we limit to 48 readings (4 hours) per scenario for speed
# Remove max_readings=48 to run the full 24-hour simulation
MAX_READINGS = 10  # 4 hours; set to None for full 24h

all_results = {}
scenario_times = {}

for scenario in CGMSimulator.SCENARIOS:
    print('\n' + '─'*50)
    print(f'▶ Running: {scenario}')
    print('─'*50)
    
    t0 = time.time()
    results = run_simulation(
        scenario=scenario,
        patient_name='alice',
        verbose=False,  # suppress per-reading output in Jupyter
        max_readings=MAX_READINGS,
    )
    elapsed = time.time() - t0
    
    all_results[scenario] = results
    scenario_times[scenario] = elapsed
    print(f'✅ {scenario}: {len(results)} readings in {elapsed:.1f}s ({elapsed/len(results)*1000:.0f}ms/reading)')

print(f'\n🎉 All scenarios complete!')

---
# 🏥 Section 4 — Results Analysis

Let's analyse the simulation results with detailed visualisations.

In [ ]:
# 4.1 Convert results to DataFrames
dfs = {}
for scenario, results in all_results.items():
    df = pd.DataFrame(results)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['hour'] = df['timestamp'].dt.hour + df['timestamp'].dt.minute / 60
    dfs[scenario] = df

print('DataFrames created:')
for scenario, df in dfs.items():
    print(f'  {scenario}: {len(df)} rows × {len(df.columns)} columns')

# Show sample
print('\nSample columns:', list(dfs['normal_day'].columns))
dfs['hypoglycemia_episode'][['timestamp', 'glucose', 'trend', 'alert_level', 'safety_score', 'caregiver_notified']].head(10)

In [ ]:
# 4.2 Glucose traces with alert overlays
fig, axes = plt.subplots(len(all_results), 1, figsize=(14, 4*len(all_results)), sharex=False)
if len(all_results) == 1:
    axes = [axes]

alert_colors = {
    'none': '#4CAF50',
    'low': '#8BC34A',
    'medium': '#FF9800',
    'high': '#FF5722',
    'critical': '#F44336',
}

for ax, (scenario, df) in zip(axes, dfs.items()):
    # Background shading for target range
    ax.axhspan(70, 180, alpha=0.08, color='green')
    ax.axhline(70, color='orange', linestyle='--', linewidth=1, alpha=0.6)
    ax.axhline(54, color='red', linestyle='--', linewidth=1, alpha=0.6)
    ax.axhline(180, color='orange', linestyle='--', linewidth=1, alpha=0.4)
    
    # Plot glucose trace coloured by alert level
    for i in range(len(df) - 1):
        color = alert_colors.get(df.iloc[i]['alert_level'], '#4CAF50')
        ax.plot(df.iloc[i:i+2]['hour'], df.iloc[i:i+2]['glucose'], 
                color=color, linewidth=2)
    
    # Mark caregiver notifications
    notified = df[df['caregiver_notified'] == True]
    if len(notified) > 0:
        ax.scatter(notified['hour'], notified['glucose'], 
                   marker='*', s=200, color='red', zorder=10, 
                   label=f'Caregiver notified ({len(notified)}x)')
        ax.legend(loc='upper right', fontsize=8)
    
    ax.set_ylabel('Glucose (mg/dL)', fontsize=9)
    ax.set_ylim(30, 320)
    ax.set_title(f'{scenario.replace("_", " ").title()}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Stats
    tir = (df['glucose'].between(70, 180)).mean() * 100
    tbr = (df['glucose'] < 70).mean() * 100
    ax.text(0.02, 0.95, f'TIR: {tir:.0f}% | TBR: {tbr:.0f}%', 
            transform=ax.transAxes, fontsize=8, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

axes[-1].set_xlabel('Hour of Day', fontsize=10)

# Alert color legend
legend_patches = [mpatches.Patch(color=c, label=f'Alert: {l}') 
                  for l, c in alert_colors.items()]
fig.legend(handles=legend_patches, loc='lower center', ncol=5, 
           fontsize=9, bbox_to_anchor=(0.5, -0.02))

fig.suptitle('GAS Simulation Results — Glucose Traces with Alert Levels', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/simulation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to figures/simulation_results.png')

In [ ]:
# 4.3 Alert accuracy metrics
print('=== Alert Distribution by Scenario ===')
print()

alert_summary = []
for scenario, df in dfs.items():
    counts = df['alert_level'].value_counts()
    total = len(df)
    row = {'scenario': scenario, 'total': total}
    for level in ['none', 'low', 'medium', 'high', 'critical']:
        count = counts.get(level, 0)
        row[level] = count
        row[f'{level}_pct'] = count / total * 100
    row['caregiver_notifications'] = df['caregiver_notified'].sum()
    row['avg_glucose'] = df['glucose'].mean()
    row['tir'] = df['glucose'].between(70, 180).mean() * 100
    row['tbr'] = (df['glucose'] < 70).mean() * 100
    row['tar'] = (df['glucose'] > 180).mean() * 100
    alert_summary.append(row)

df_summary = pd.DataFrame(alert_summary)
display_cols = ['scenario', 'total', 'avg_glucose', 'tir', 'tbr', 'tar', 
                'none', 'low', 'medium', 'high', 'critical', 'caregiver_notifications']
print(df_summary[display_cols].to_string(index=False, float_format='{:.1f}'.format))

In [ ]:
# 4.4 Alert distribution stacked bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Alert distribution
ax = axes[0]
scenarios = df_summary['scenario'].tolist()
x = np.arange(len(scenarios))
width = 0.6

bottoms = np.zeros(len(scenarios))
for level, color in alert_colors.items():
    values = df_summary[f'{level}_pct'].values
    ax.bar(x, values, width, bottom=bottoms, label=f'Alert: {level}', color=color, alpha=0.85)
    bottoms += values

ax.set_xticks(x)
ax.set_xticklabels([s.replace('_', '\n') for s in scenarios], fontsize=9)
ax.set_ylabel('Percentage of Readings (%)')
ax.set_title('Alert Level Distribution by Scenario', fontweight='bold')
ax.legend(loc='upper right', fontsize=8)
ax.axhline(100, color='black', linewidth=0.5)
ax.set_ylim(0, 110)
ax.grid(True, axis='y', alpha=0.3)

# Right: Time in Range comparison
ax2 = axes[1]
metrics = ['tir', 'tbr', 'tar']
metric_labels = ['Time in Range\n(70–180 mg/dL)', 'Time Below Range\n(< 70 mg/dL)', 'Time Above Range\n(> 180 mg/dL)']
metric_colors = ['#4CAF50', '#F44336', '#FF9800']
metric_targets = [70, 4, None]  # ADA targets

x2 = np.arange(len(scenarios))
bar_width = 0.25

for i, (metric, label, color, target) in enumerate(zip(metrics, metric_labels, metric_colors, metric_targets)):
    vals = df_summary[metric].values
    bars = ax2.bar(x2 + i*bar_width, vals, bar_width, label=label, color=color, alpha=0.8)
    if target is not None:
        ax2.axhline(target, color=color, linestyle='--', linewidth=1, alpha=0.6)

ax2.set_xticks(x2 + bar_width)
ax2.set_xticklabels([s.replace('_', '\n') for s in scenarios], fontsize=9)
ax2.set_ylabel('Percentage (%)')
ax2.set_title('Time in/Below/Above Range by Scenario\n(dashed = ADA target)', fontweight='bold')
ax2.legend(loc='upper right', fontsize=8)
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/alert_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to figures/alert_metrics.png')

In [ ]:
# 4.5 Safety Critic analysis
print('=== Safety Critic Analysis ===')
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Safety score distribution
ax = axes[0]
all_scores = []
for scenario, df in dfs.items():
    scores = df['safety_score'].dropna().tolist()
    all_scores.extend(scores)
    ax.hist(scores, bins=20, alpha=0.5, label=scenario.replace('_', ' '), density=True)

ax.axvline(0.7, color='red', linestyle='--', linewidth=2, label='Approval threshold (0.7)')
ax.set_xlabel('Safety Score')
ax.set_ylabel('Density')
ax.set_title('Safety Score Distribution\n(all scenarios)', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Approval rate by scenario
ax2 = axes[1]
approval_rates = []
for scenario, df in dfs.items():
    rate = df['safety_approved'].mean() * 100
    approval_rates.append(rate)
    print(f'  {scenario}: {rate:.1f}% approved, avg score: {df["safety_score"].mean():.3f}')

bars = ax2.bar(range(len(dfs)), approval_rates, 
               color=[scenario_colors.get(s, '#607D8B') for s in dfs.keys()],
               alpha=0.85)
ax2.set_xticks(range(len(dfs)))
ax2.set_xticklabels([s.replace('_', '\n') for s in dfs.keys()], fontsize=9)
ax2.set_ylabel('Approval Rate (%)')
ax2.set_title('Safety Critic Approval Rate\nby Scenario', fontweight='bold')
ax2.set_ylim(0, 110)
ax2.axhline(100, color='green', linestyle='--', alpha=0.5)
ax2.grid(True, axis='y', alpha=0.3)

for bar, rate in zip(bars, approval_rates):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'{rate:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/safety_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nFigure saved to figures/safety_analysis.png')

In [ ]:
# 4.6 Latency breakdown by scenario
print('=== Latency Analysis ===')
print()

latency_data = []
for scenario, elapsed in scenario_times.items():
    n = len(all_results[scenario])
    ms_per_reading = elapsed / n * 1000
    latency_data.append({
        'scenario': scenario,
        'total_seconds': elapsed,
        'n_readings': n,
        'ms_per_reading': ms_per_reading,
    })
    print(f'  {scenario}: {n} readings in {elapsed:.2f}s = {ms_per_reading:.0f}ms/reading')

df_latency = pd.DataFrame(latency_data)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [scenario_colors.get(s, '#607D8B') for s in df_latency['scenario']]
bars = ax.bar(range(len(df_latency)), df_latency['ms_per_reading'], color=colors, alpha=0.85)

ax.set_xticks(range(len(df_latency)))
ax.set_xticklabels([s.replace('_', '\n') for s in df_latency['scenario']], fontsize=10)
ax.set_ylabel('Latency (ms per reading)')
ax.set_title('GAS Pipeline Latency by Scenario\n(monitor → advisor → safety → caregiver)', 
             fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

for bar, row in zip(bars, df_latency.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{row.ms_per_reading:.0f}ms', ha='center', va='bottom', fontsize=9)

# Add note about test mode
mode_note = 'Mock mode (no API calls)' if TEST_MODE else 'Live LLM calls'
ax.text(0.98, 0.95, f'Mode: {mode_note}', transform=ax.transAxes,
        ha='right', va='top', fontsize=9, style='italic',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('figures/latency_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to figures/latency_analysis.png')

In [ ]:
# 4.7 Sample recommendations from the Advisor agent
print('=== Sample Advisor Recommendations ===')
print()

for scenario, df in dfs.items():
    # Find the most critical reading
    critical = df[df['alert_level'].isin(['high', 'critical'])]
    if len(critical) > 0:
        row = critical.iloc[0]
        print(f'Scenario: {scenario}')
        print(f'  Glucose: {row["glucose"]:.1f} mg/dL | Trend: {row["trend"]} | Alert: {row["alert_level"].upper()}')
        print(f'  Recommendation: {row["recommendation"]}')
        print(f'  Safety Score: {row["safety_score"]:.2f} | Approved: {row["safety_approved"]}')
        print(f'  Caregiver Notified: {row["caregiver_notified"]}')
        print()

---
# ✅ Section 5 — Extending GAS

The Guardian Angel System is designed to be extended. Here's how to add new capabilities.

In [ ]:
# 5.1 Adding a new agent: Nutritionist Agent
print('=== Example: Adding a Nutritionist Agent ===')
print()

# Step 1: Define the agent node function
def nutritionist_node(state: GASState) -> dict:
    """
    Nutritionist Agent — Suggests meal adjustments based on glucose patterns.
    
    Activates when:
    - Post-meal spike detected (glucose > 200 mg/dL with 'meal' in notes)
    - Glucose is in range but trending up after a meal
    
    Returns meal-specific recommendations grounded in carbohydrate counting.
    """
    reading = state['current_reading']
    patient = state['patient']
    alert_reason = state.get('alert_reason', '')
    
    glucose = reading['value_mgdl']
    trend = reading['trend']
    notes = reading.get('notes', '')
    
    # Only activate for post-meal situations
    is_post_meal = any(kw in notes.lower() for kw in ['meal', 'lunch', 'dinner', 'breakfast'])
    
    if not is_post_meal and glucose < 200:
        return {
            'agent_messages': ['[Nutritionist] No meal-related intervention needed'],
        }
    
    # Generate meal recommendation
    carb_ratio = patient.get('carb_ratio', 10)
    
    if glucose > 200 and trend in ('rising', 'rising_fast'):
        # Estimate carbs consumed based on glucose rise
        estimated_carbs = (glucose - 120) / (1 / carb_ratio * 50)  # rough estimate
        recommendation = (
            f'Post-meal spike detected ({glucose:.0f} mg/dL). '
            f'For future meals: pre-bolus insulin 15–20 min before eating. '
            f'Consider reducing carbohydrate portion by ~20g. '
            f'Your carb ratio is 1:{carb_ratio} (1 unit covers {carb_ratio}g carbs).'
        )
    else:
        recommendation = (
            f'Glucose is {glucose:.0f} mg/dL after meal. '
            f'Current management appears appropriate. '
            f'Continue monitoring for 2 hours post-meal.'
        )
    
    msg = f'[Nutritionist] Meal recommendation: {recommendation[:80]}...'
    
    return {
        'agent_messages': [msg],
        # Could add: 'meal_recommendation': recommendation
    }

# Step 2: Test the new agent
test_state = {
    'patient': PATIENT_PROFILES['alice'].to_dict(),
    'readings_history': [],
    'current_reading': {
        'timestamp': '2025-01-15T13:30:00+00:00',
        'value_mgdl': 245.0,
        'trend': 'rising',
        'source': 'CGM',
        'notes': 'large lunch',
    },
    'alert_level': 'high',
    'alert_reason': 'Post-meal spike',
    'recommendation': '',
    'safety_score': 1.0,
    'safety_approved': False,
    'caregiver_notified': False,
    'agent_messages': [],
    'next_agent': 'nutritionist',
    'iteration': 0,
}

result = nutritionist_node(test_state)
print('Nutritionist Agent output:')
for msg in result['agent_messages']:
    print(f'  {msg}')

print()
print('To add to the graph:')
print('''
  builder.add_node("nutritionist", nutritionist_node)
  # Insert between advisor and safety:
  builder.add_edge("advisor", "nutritionist")
  builder.add_edge("nutritionist", "safety")
  # Remove: builder.add_edge("advisor", "supervisor")
''')

In [ ]:
# 5.2 Adding a new tool: Insulin Pump Control
print('=== Example: Adding an Insulin Pump Tool ===')
print()

# In production, this would call the pump's API
# Here we show the interface pattern

def adjust_basal_rate(rate_units_per_hour: float, duration_hours: float = 2.0) -> dict:
    """
    Adjust insulin pump basal rate (simulated).
    
    Parameters
    ----------
    rate_units_per_hour : float
        New basal rate in units per hour.
    duration_hours : float
        Duration of the temporary rate change.
    
    Returns
    -------
    dict
        Pump response with status and confirmation.
    
    Safety constraints
    ------------------
    - Maximum rate: 5.0 U/hr (hard limit)
    - Minimum rate: 0.0 U/hr (suspend)
    - Requires glucose > 80 mg/dL to increase rate
    """
    # Safety bounds
    MAX_RATE = 5.0
    rate_units_per_hour = max(0.0, min(MAX_RATE, rate_units_per_hour))
    
    # Simulate pump API call
    return {
        'status': 'success',
        'new_rate': rate_units_per_hour,
        'duration_hours': duration_hours,
        'effective_until': f'+{duration_hours:.0f}h from now',
        'pump_id': 'OMNIPOD-5-SIM-001',
        'confirmation_code': f'PUMP-{int(rate_units_per_hour*100):04d}',
    }

# Test the tool
print('Test: Increase basal rate for dawn phenomenon')
result = adjust_basal_rate(rate_units_per_hour=1.5, duration_hours=3.0)
print(f'  Result: {result}')

print()
print('Test: Suspend pump during exercise')
result = adjust_basal_rate(rate_units_per_hour=0.0, duration_hours=1.0)
print(f'  Result: {result}')

print()
print('To integrate with LangChain tools:')
print('''
  from langchain_core.tools import tool
  
  @tool
  def adjust_basal_rate_tool(rate: float, duration: float = 2.0) -> str:
      """Adjust insulin pump basal rate. Only use when glucose > 80 mg/dL."""
      result = adjust_basal_rate(rate, duration)
      return f"Pump adjusted to {rate} U/hr for {duration}h. Code: {result['confirmation_code']}"
''')

In [ ]:
# 5.3 Production deployment architecture
print('=== Production Deployment Architecture ===')
print()

deployment_architecture = '''
Production GAS Deployment
=========================

┌─────────────────────────────────────────────────────────────────┐
│                    Kubernetes Cluster                           │
│                                                                 │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────────────┐   │
│  │  GAS API     │  │  GAS Worker  │  │  GAS Notifier        │   │
│  │  (FastAPI)   │  │  (LangGraph) │  │  (Email/SMS/Push)    │   │
│  │  Port: 8000  │  │  3 replicas  │  │  2 replicas          │   │
│  └──────┬───────┘  └──────┬───────┘  └──────────────────────┘   │
│         │                 │                                     │
│         ▼                 ▼                                     │
│  ┌──────────────────────────────────────────────────────────┐   │
│  │                   Redis Streams                          │   │
│  │  gas:readings  │  gas:alerts  │  gas:notifications       │   │
│  └──────────────────────────────────────────────────────────┘   │
│                                                                 │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────────────┐   │
│  │  Qdrant      │  │  PostgreSQL  │  │  Prometheus/Grafana  │   │
│  │  (RAG index) │  │  (audit log) │  │  (observability)     │   │
│  └──────────────┘  └──────────────┘  └──────────────────────┘   │
└─────────────────────────────────────────────────────────────────┘

Docker Compose (development):
  docker compose up -d
  # Starts: gas-api, gas-worker, qdrant, redis, postgres

Kubernetes (production):
  kubectl apply -f k8s/
  # Deploys with HPA, PodDisruptionBudget, NetworkPolicy

Health check:
  curl http://localhost:8000/health
  # {"status": "healthy", "agents": 5, "kb_docs": 15, "llm_backend": "openai"}
'''
print(deployment_architecture)

---
# ✅ Section 6 — Clinical Considerations

A responsible AI system must be honest about its capabilities and limitations.

In [ ]:
# 6.1 What GAS does well
print('=== What GAS Does Well ===')
print()

strengths = [
    ('24/7 Continuous Monitoring', 
     'GAS never sleeps, never gets fatigued, and processes every 5-minute CGM reading '
     'with consistent attention. Human caregivers cannot maintain this level of vigilance.'),
    
    ('Consistent Guideline Application',
     'Every recommendation is grounded in ADA guidelines retrieved from the knowledge base. '
     'GAS applies the same rules every time, without the variability of human judgment.'),
    
    ('Rapid Escalation',
     'Critical alerts (glucose < 54 mg/dL) trigger caregiver notification in < 1 second. '
     'This is faster than any human monitoring system.'),
    
    ('Audit Trail',
     'Every agent decision is logged in agent_messages. This creates a complete audit trail '
     'for clinical review, quality improvement, and regulatory compliance.'),
    
    ('Safety-First Architecture',
     'The Safety Critic validates every recommendation before delivery. '
     'Hard rules prevent dangerous advice (e.g., insulin during hypoglycemia).'),
    
    ('Personalisation',
     'Each patient has a unique profile (ISF, carb ratio, medications, targets). '
     'Recommendations are tailored to the individual, not generic.'),
]

for title, description in strengths:
    print(f'✅ {title}')
    print(f'   {description}')
    print()

In [ ]:
# 6.2 What GAS cannot do
print('=== What GAS Cannot Do ===')
print()

limitations = [
    ('Replace Clinical Judgment',
     'GAS cannot account for the full clinical picture: comorbidities, recent illness, '
     'stress, medication changes, or the nuanced judgment of an experienced endocrinologist.'),
    
    ('Guarantee Recommendation Accuracy',
     'LLMs can hallucinate. Even with RAG grounding, the Advisor agent may generate '
     'incorrect recommendations. The Safety Critic catches obvious errors but cannot '
     'catch all subtle mistakes.'),
    
    ('Handle Novel Clinical Situations',
     'GAS is trained on common diabetes scenarios. Rare conditions (e.g., adrenal '
     'insufficiency, unusual drug interactions) may not be handled correctly.'),
    
    ('Verify CGM Accuracy',
     'GAS trusts the CGM readings it receives. Sensor errors, calibration issues, '
     'or compression artifacts can lead to false alerts or missed events.'),
    
    ('Act Without Human Oversight',
     'GAS is a decision support system, not an autonomous medical device. '
     'All recommendations require human review before action.'),
    
    ('Provide Emergency Medical Care',
     'For life-threatening emergencies, GAS escalates to caregivers and recommends '
     'calling emergency services. It cannot administer glucagon or call 911 itself.'),
]

for title, description in limitations:
    print(f'❌ {title}')
    print(f'   {description}')
    print()

In [ ]:
# 6.3 Path to clinical validation
print('=== Path to Clinical Validation ===')
print()

validation_path = [
    ('Phase 1: Technical Validation',
     ['Unit tests for all safety rules (100% coverage)',
      'Retrospective validation on historical CGM datasets',
      'Adversarial testing (red-team the Safety Critic)',
      'Latency benchmarking (< 5s end-to-end for critical alerts)']),
    
    ('Phase 2: Clinical Expert Review',
     ['Endocrinologist review of 500 sample recommendations',
      'Comparison with standard-of-care guidelines',
      'Identification of failure modes and edge cases',
      'Refinement of safety rules based on expert feedback']),
    
    ('Phase 3: IRB-Approved Pilot Study',
     ['Prospective observational study (n=50 patients, 3 months)',
      'Primary endpoint: Time in Range improvement',
      'Secondary endpoints: Hypoglycemia events, caregiver satisfaction',
      'Safety monitoring: Independent Data Safety Monitoring Board']),
    
    ('Phase 4: Regulatory Pathway',
     ['FDA 510(k) clearance as Software as a Medical Device (SaMD)',
      'IEC 62304 software lifecycle compliance',
      'ISO 14971 risk management documentation',
      'Post-market surveillance plan']),
    
    ('Phase 5: Commercialisation',
     ['HIPAA compliance and data privacy audit',
      'Integration with major CGM platforms (Dexcom, Abbott)',
      'EHR integration (Epic, Cerner)',
      'Reimbursement pathway (CPT codes for remote monitoring)']),
]

for phase, steps in validation_path:
    print(f'📋 {phase}')
    for step in steps:
        print(f'   • {step}')
    print()

In [ ]:
# 6.4 Ethical considerations
print('=== Ethical Considerations ===')
print()

ethics = [
    ('Autonomy',
     'Patients must retain control over their own care. GAS provides recommendations, '
     'not commands. Patients can always override or ignore GAS suggestions. '
     'Informed consent is required for any clinical deployment.'),
    
    ('Beneficence & Non-Maleficence',
     'The Safety Critic embodies the principle of "first, do no harm". '
     'Hard safety rules prevent dangerous advice. But we must also consider '
     'alert fatigue — too many false alarms can cause patients to ignore real emergencies.'),
    
    ('Justice & Equity',
     'AI systems trained on biased data can perpetuate health disparities. '
     'GAS must be validated across diverse patient populations (age, race, '
     'socioeconomic status, diabetes type) to ensure equitable performance.'),
    
    ('Privacy & Data Security',
     'Glucose data is highly sensitive health information. GAS must comply with '
     'HIPAA, GDPR, and applicable data protection laws. Patient data must never '
     'be used to train commercial LLMs without explicit consent.'),
    
    ('Transparency & Explainability',
     'Patients and caregivers deserve to understand why GAS made a recommendation. '
     'The agent_messages audit trail provides explainability. '
     'The RAG system shows which guidelines informed each recommendation.'),
    
    ('Accountability',
     'When GAS makes an error, who is responsible? The developer? The clinician? '
     'The patient? Clear accountability frameworks must be established before '
     'clinical deployment. GAS should always be positioned as a tool, not a decision-maker.'),
]

for principle, discussion in ethics:
    print(f'⚖️  {principle}')
    print(f'   {discussion}')
    print()

In [ ]:
# 6.5 Final summary visualisation
fig = plt.figure(figsize=(14, 8))
gs = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.3)

# 1. Overall TIR comparison
ax1 = fig.add_subplot(gs[0, 0])
scenarios_short = [s.replace('_', '\n') for s in df_summary['scenario']]
tir_values = df_summary['tir'].values
colors_bar = [scenario_colors.get(s, '#607D8B') for s in df_summary['scenario']]
ax1.bar(range(len(scenarios_short)), tir_values, color=colors_bar, alpha=0.85)
ax1.axhline(70, color='green', linestyle='--', linewidth=2, label='ADA target (70%)')
ax1.set_xticks(range(len(scenarios_short)))
ax1.set_xticklabels(scenarios_short, fontsize=7)
ax1.set_ylabel('TIR (%)')
ax1.set_title('Time in Range\nby Scenario', fontweight='bold', fontsize=10)
ax1.legend(fontsize=7)
ax1.grid(True, axis='y', alpha=0.3)

# 2. Caregiver notifications
ax2 = fig.add_subplot(gs[0, 1])
notif_values = df_summary['caregiver_notifications'].values
ax2.bar(range(len(scenarios_short)), notif_values, color=colors_bar, alpha=0.85)
ax2.set_xticks(range(len(scenarios_short)))
ax2.set_xticklabels(scenarios_short, fontsize=7)
ax2.set_ylabel('Count')
ax2.set_title('Caregiver Notifications\nby Scenario', fontweight='bold', fontsize=10)
ax2.grid(True, axis='y', alpha=0.3)

# 3. Average glucose
ax3 = fig.add_subplot(gs[0, 2])
avg_values = df_summary['avg_glucose'].values
ax3.bar(range(len(scenarios_short)), avg_values, color=colors_bar, alpha=0.85)
ax3.axhline(154, color='orange', linestyle='--', linewidth=1.5, label='~HbA1c 7% (154 mg/dL)')
ax3.set_xticks(range(len(scenarios_short)))
ax3.set_xticklabels(scenarios_short, fontsize=7)
ax3.set_ylabel('mg/dL')
ax3.set_title('Average Glucose\nby Scenario', fontweight='bold', fontsize=10)
ax3.legend(fontsize=7)
ax3.grid(True, axis='y', alpha=0.3)

# 4. GAS system summary text
ax4 = fig.add_subplot(gs[1, :])
ax4.axis('off')

total_readings = sum(len(r) for r in all_results.values())
total_alerts = sum(df_summary['high'] + df_summary['critical'])
total_notifications = df_summary['caregiver_notifications'].sum()
avg_tir = df_summary['tir'].mean()

sep = '─' * 70
summary_text = (
    f'Guardian Angel System — Capstone Simulation Summary\n'
    f'{sep}\n'
    f'Scenarios run: {len(all_results)}  |  '
    f'Total readings processed: {total_readings}  |  '
    f'High/Critical alerts: {total_alerts}\n'
    f'Caregiver notifications sent: {total_notifications}  |  '
    f'Average TIR across scenarios: {avg_tir:.1f}%  |  '
    f'LLM backend: {"Mock (TEST_MODE)" if TEST_MODE else "Live"}\n'
    '\n'
    '"The Guardian Angel System demonstrates that multi-agent AI can provide\n'
    f' continuous, evidence-based, safety-validated support for diabetic patients.\n'
    f' This is the culmination of 28 chapters of agentic AI development."'
)

ax4.text(0.5, 0.5, summary_text, transform=ax4.transAxes,
         ha='center', va='center', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9),
         fontfamily='monospace')

fig.suptitle('★ GAS Capstone — Final Results Dashboard', 
             fontsize=14, fontweight='bold')

plt.savefig('figures/capstone_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to figures/capstone_dashboard.png')

In [ ]:
# Final message
print('''
╔══════════════════════════════════════════════════════════════════════╗
║                                                                      ║
║   🎉  Congratulations! You've completed the GAS Capstone.            ║
║                                                                      ║
║   You've built a complete multi-agent AI system that:                ║
║                                                                      ║
║   ✅ Monitors continuous glucose data (Monitor Agent)                ║
║   ✅ Generates RAG-grounded recommendations (Advisor Agent)          ║
║   ✅ Validates safety with constitutional rules (Safety Critic)      ║
║   ✅ Escalates to caregivers when needed (Caregiver Notifier)        ║
║   ✅ Orchestrates everything with LangGraph (Supervisor)             ║
║                                                                      ║
║   This system integrates:                                            ║
║   • LangGraph StateGraph orchestration (Ch. 15)                      ║
║   • RAG with hybrid retrieval + sentence-transformers (Ch. 16)       ║
║   • Constitutional AI safety rules (Ch. 8)                           ║
║   • Multi-agent coordination patterns (Ch. 16-17)                    ║
║   • Production-ready LLM client (Ch. 2, 11)                          ║
║                                                                      ║
║   The Guardian Angel watches over its patients. 🏥                   ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
''')